# ResNet50 Transfer Learning + Fine-Tuning + Data Augmentation — ISIC 2019 Skin Lesion Classification

## Experiment 05

**Architecture:** ResNet50  
**Training Strategy:** Transfer Learning + Fine-Tuning + Data Augmentation  
**Pretrained Weights:** ImageNet (`IMAGENET1K_V2`)  
**Data Augmentation:** Yes, on training images only  
**Dataset:** ISIC 2019  
**Number of Classes:** 8  
**Model Input Size:** 224 × 224  
**Cached Image Size:** 256 × 256  
**Framework:** PyTorch and TorchVision  
**Supported Hardware:** NVIDIA GPUs, Apple Silicon, and CPUs  
**Supported Compute Devices:** CUDA, MPS, and CPU

---

## Objective

Investigate whether modest image augmentation improves ResNet50 transfer learning on eight-class lesion classification. Experiment 03 provides the unaugmented ResNet50 comparison. Experiment 05 adds dynamic augmentation during training and keeps evaluation preprocessing deterministic.

**Research question.** How do training-only flips, rotations, and mild color adjustments affect validation macro F1 and held-out class-wise performance?

## Stage 1 — Transfer Learning / Feature Extraction

- Initialize ResNet50 with ImageNet weights.
- Replace the ImageNet fully connected layer with an eight-output linear layer.
- Freeze the backbone and its BatchNorm running statistics while training the new classification layer.

## Stage 2 — Fine-Tuning

- Restore this run's best Stage 1 checkpoint by validation macro F1.
- Unfreeze the backbone and allow BatchNorm statistics to update.
- Fine-tune the entire network with a lower learning rate for the backbone than for the classification layer.

---

## Data Augmentation Strategy

- Horizontal and vertical flips use a probability of 0.5 each to vary lesion orientation.
- Bilinear rotation up to ±20° adds small orientation changes. Reflection padding keeps constant-fill corners outside the crop back to 256 × 256.
- Mild color jitter varies brightness and contrast by 0.1 and saturation by 0.05 to represent small image-acquisition differences.

These transforms are applied on the fly to the training split in both stages. All splits use the same final resize, tensor conversion, and normalization. Validation and test use those deterministic steps alone.

## Experimental Control

Reuse the permanent lesion-aware splits, the 256 × 256 JPEG cache, the eight-class label mapping, 224 × 224 inputs, ImageNet normalization, and weighted cross-entropy with weights computed from training labels. Checkpoint selection uses validation macro F1 before the held-out test evaluation.

The batch size, optimizer, learning rates, epoch limits, patience, and sample-count-weighted averaging of batch losses follow Experiment 03. CUDA mixed precision is off by default to match its full-precision policy. CUDA and MPS can still differ numerically.

Experiments 04 and 05 compare both stages for final checkpoint selection and favor Stage 1 on an exact tie. Experiment 03 reported its best Stage 2 checkpoint. Cross-experiment comparisons should account for this selection difference and for hardware differences.

## Research Comparisons

Experiment 03 provides the unaugmented ResNet50 comparison. Experiment 06 applies the same augmentation to DenseNet121. For the architecture comparison, account for the different explicit TF32 policies, pretrained weight variants, and validation-loss aggregation.

## Completed cloud run

See the [published aggregate results](../results/cloud/) and [research audit](../docs/research-audit.md). The recorded RTX 3090 runs used four DataLoader workers and AMP disabled. This notebook uses zero workers for local Jupyter compatibility. Use an empty output directory for training.

## A. Project Setup

In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50
from torchvision.transforms import InterpolationMode

print("=" * 60)
print("LIBRARY VERSIONS")
print("=" * 60)
print("PyTorch     :", torch.__version__)
print("TorchVision :", torchvision.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

In [ ]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "dataset"
SPLIT_DIR = DATA_DIR / "splits"
CACHE_DIR = DATA_DIR / "ISIC_2019_256"
BASELINE_DIR = PROJECT_DIR / "results"
OUTPUT_DIR = PROJECT_DIR / "runs" / "resnet50_transfer_aug"
MODEL_DIR = OUTPUT_DIR
RESULTS_DIR = OUTPUT_DIR

TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"


def artifact(filename):
    directory = MODEL_DIR if filename.endswith(".pth") else RESULTS_DIR
    return directory / f"resnet50_transfer_aug_{filename}"


print("Project directory :", PROJECT_DIR)
print("Permanent splits  :", SPLIT_DIR)
print("Existing cache    :", CACHE_DIR)
print("Model output      :", MODEL_DIR)
print("Results output    :", RESULTS_DIR)

In [ ]:
# ============================================================
# Experiment Configuration
# ============================================================

USE_CUDA_AMP = False
BATCH_SIZE = 32
NUM_WORKERS = 0  # Mac notebook default. The cloud run used 4.
RANDOM_SEED = 42
CACHE_SIZE = 256
IMAGE_SIZE = 224
CLASS_CODES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES = len(CLASS_CODES)
CLASS_IDS = list(range(NUM_CLASSES))

HORIZONTAL_FLIP = 0.5
VERTICAL_FLIP = 0.5
ROTATION_DEGREES = 20
BRIGHTNESS = 0.1
CONTRAST = 0.1
SATURATION = 0.05

STAGE1_EPOCHS = 10
STAGE2_EPOCHS = 15
STAGE1_PATIENCE = 3
STAGE2_PATIENCE = 4
STAGE1_LR = 1e-3
BACKBONE_LR = 1e-5
CLASSIFIER_LR = 1e-4
WEIGHT_DECAY = 1e-4
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2

In [ ]:
# ============================================================
# Reproducibility Configuration
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print("Configured random seed:", RANDOM_SEED)

In [ ]:
# ============================================================
# Device Configuration
# ============================================================

if torch.cuda.is_available():
    device = torch.device("cuda")
    hardware = torch.cuda.get_device_name(device)
    # Keep CUDA's float32 arithmetic policy close to the MPS baseline.
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    hardware = "Apple Silicon (MPS)"
else:
    device = torch.device("cpu")
    hardware = "CPU"

on_cuda = device.type == "cuda"
use_amp = on_cuda and USE_CUDA_AMP
print(f"Device: {device} ({hardware}) | CUDA mixed precision: {use_amp}")

In [ ]:
# ============================================================
# Training Recovery Settings
# ============================================================

RESUME_TRAINING = False

print("Resume training:", RESUME_TRAINING)
print("CUDA mixed precision active:", use_amp)

## Dataset Preparation

Load the permanent ISIC 2019 splits and use the existing resized JPEG cache. Inspect class labels, image paths, and one processed batch before model preparation.

In [ ]:
# ============================================================
# Load Permanent Dataset Splits
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

splits = {"train": train_df, "val": val_df, "test": test_df}
expected_sizes = {"train": 17705, "val": 3818, "test": 3808}

print("=" * 60)
print("PERMANENT DATASET SPLITS")
print("=" * 60)
print(f"Training images   : {len(train_df):,}")
print(f"Validation images : {len(val_df):,}")
print(f"Test images       : {len(test_df):,}")
print(f"Total images      : {sum(len(frame) for frame in splits.values()):,}")
print("\nTraining columns:", train_df.columns.tolist())
display(train_df.head(3))

In [ ]:
# ============================================================
# Build Class Information Table
# ============================================================

class_info = (
    train_df[["class_index", "class_code", "class_name"]]
    .drop_duplicates()
    .sort_values("class_index")
    .reset_index(drop=True)
)

if class_info["class_index"].tolist() != CLASS_IDS or class_info["class_name"].isna().any():
    raise ValueError("The training split must define one name for each ISIC class index.")
if class_info["class_code"].tolist() != CLASS_CODES:
    raise ValueError("The training split does not match the original ISIC class order.")

class_names = class_info["class_name"].tolist()
class_to_index = dict(zip(CLASS_CODES, CLASS_IDS))
name_by_index = dict(zip(CLASS_IDS, class_names))

print("=" * 60)
print("CLASS INFORMATION")
print("=" * 60)
print("Number of classes:", NUM_CLASSES)
display(class_info)

class_distribution = pd.DataFrame({
    "Train": train_df["class_code"].value_counts(),
    "Validation": val_df["class_code"].value_counts(),
    "Test": test_df["class_code"].value_counts(),
}).reindex(CLASS_CODES).fillna(0).astype(int)
print("\nImages per class and split:")
display(class_distribution)

In [ ]:
# ============================================================
# Configure Existing 256x256 Image Cache
# ============================================================

print("=" * 60)
print("RESIZED IMAGE CACHE")
print("=" * 60)
print("Cache directory :", CACHE_DIR)
print("Cached dimensions :", (CACHE_SIZE, CACHE_SIZE))
print("Expected JPEGs  :", sum(expected_sizes.values()))

if not CACHE_DIR.is_dir():
    raise FileNotFoundError(f"Mount the existing resized image cache at {CACHE_DIR}.")

In [ ]:
# ============================================================
# Verify Image Cache
# ============================================================

cached_images = {path.name for path in CACHE_DIR.glob("*.jpg") if path.is_file()}
missing_counts = {}

print("=" * 60)
print("IMAGE CACHE VERIFICATION")
print("=" * 60)
print(f"Cached JPEGs : {len(cached_images):,}")
for split_name, frame in splits.items():
    missing_counts[split_name] = sum(f"{image_id}.jpg" not in cached_images for image_id in frame["image"])
    print(f"Missing {split_name:5s} JPEGs: {missing_counts[split_name]:,}")

if any(missing_counts.values()):
    raise FileNotFoundError("The cache is missing images referenced by the permanent splits.")

In [ ]:
# ============================================================
# Add Cached Image Paths
# ============================================================

# Build paths from the image IDs so the original machine's paths are not needed on a GPU service.
for frame in splits.values():
    frame["cached_image_path"] = [str(CACHE_DIR / f"{image_id}.jpg") for image_id in frame["image"]]

print("Example cached image:", train_df.loc[0, "cached_image_path"])
display(train_df[["image", "class_code", "cached_image_path"]].head(3))

In [ ]:
# ============================================================
# Verify Dataset Consistency and Data Leakage
# ============================================================

for split_name, frame in splits.items():
    if len(frame) != expected_sizes[split_name] or set(frame["class_index"]) != set(CLASS_IDS):
        raise ValueError(f"{split_name}: unexpected split size or class indices.")
    if not frame["class_code"].map(class_to_index).eq(frame["class_index"]).all():
        raise ValueError(f"{split_name}: the class codes do not match the original labels.")
    if not frame["class_index"].map(name_by_index).eq(frame["class_name"]).all():
        raise ValueError(f"{split_name}: the class names do not match the original labels.")

all_images = pd.concat([
    frame.assign(partition=split_name) for split_name, frame in splits.items()
], ignore_index=True)
if all_images["image"].isna().any() or all_images["image"].duplicated().any():
    raise ValueError("The permanent splits contain missing or repeated image IDs.")
if all_images["split_group"].isna().any():
    raise ValueError("One or more images are missing a split-group ID.")

print("=" * 60)
print("DATASET CONSISTENCY AND LEAKAGE CHECKS")
print("=" * 60)
print(f"Verified image IDs : {all_images['image'].nunique():,}")
for column in ("split_group", "lesion_id"):
    # Missing lesion IDs have their own fallback split group in the permanent manifests.
    split_counts = all_images.groupby(column)["partition"].nunique()
    overlapping_groups = int(split_counts.gt(1).sum())
    print(f"Overlapping {column:11s}: {overlapping_groups:,}")
    if overlapping_groups:
        raise ValueError(f"A {column} appears in more than one split.")

print(f"Images without a known lesion ID: {all_images['lesion_id'].isna().sum():,}")

In [ ]:
# ============================================================
# Image Preprocessing — Resize and Normalize
# ============================================================

image_mean = (0.485, 0.456, 0.406)
image_std = (0.229, 0.224, 0.225)

base_steps = [
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(image_mean, image_std),
]
evaluation_transform = transforms.Compose(base_steps)

print("Cached resolution :", (CACHE_SIZE, CACHE_SIZE))
print("Model input size  :", (IMAGE_SIZE, IMAGE_SIZE))
print("ImageNet mean     :", image_mean)
print("ImageNet std      :", image_std)
print("\nValidation and test preprocessing:")
print(evaluation_transform)

In [ ]:
# ============================================================
# Build Training-Only Augmentation
# ============================================================

rotation_padding = 56
rotation_fill = tuple(round(value * 255) for value in image_mean)

# ------------------------------------------------------------
# Geometry and Color Adjustments
# ------------------------------------------------------------
# Reflect-pad before rotation, then crop back to the cached dimensions.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(HORIZONTAL_FLIP),
    transforms.RandomVerticalFlip(VERTICAL_FLIP),
    transforms.Pad(rotation_padding, padding_mode="reflect"),
    transforms.RandomRotation(
        ROTATION_DEGREES, interpolation=InterpolationMode.BILINEAR, fill=rotation_fill
    ),
    transforms.CenterCrop(CACHE_SIZE),
    transforms.ColorJitter(brightness=BRIGHTNESS, contrast=CONTRAST, saturation=SATURATION),
    *base_steps,
])

print("Training preprocessing:")
print(train_transform)

In [ ]:
# ============================================================
# Custom ISIC Dataset
# ============================================================

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        label = int(row["class_index"])

        with Image.open(row["cached_image_path"]) as cached_image:
            image = self.transform(cached_image.convert("RGB"))

        return image, label, row["image"]

In [ ]:
# ============================================================
# Create PyTorch Dataset Objects
# ============================================================

train_dataset = ISICDataset(train_df, transform=train_transform)
val_dataset = ISICDataset(val_df, transform=evaluation_transform)
test_dataset = ISICDataset(test_df, transform=evaluation_transform)

print("=" * 60)
print("PYTORCH DATASETS")
print("=" * 60)
print(f"Training samples   : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")
print(f"Test samples       : {len(test_dataset):,}")

In [ ]:
# ============================================================
# Verify One Processed Sample
# ============================================================

# A sample check should leave the RNG used by training unchanged.
with torch.random.fork_rng(devices=[]):
    sample_image, sample_label, sample_image_id = train_dataset[0]

print("=" * 60)
print("PROCESSED SAMPLE")
print("=" * 60)
print("Image ID           :", sample_image_id)
print("Image tensor shape :", tuple(sample_image.shape))
print("Image tensor dtype :", sample_image.dtype)
print("Class index        :", sample_label)
print("Class code         :", CLASS_CODES[sample_label])
print("Class name         :", class_names[sample_label])

assert sample_image.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
assert sample_image.dtype == torch.float32
assert 0 <= sample_label < NUM_CLASSES

In [ ]:
# ============================================================
# DataLoader Configuration
# ============================================================

train_generator = torch.Generator().manual_seed(RANDOM_SEED)
loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": on_cuda,
}

print("=" * 60)
print("DATALOADER CONFIGURATION")
print("=" * 60)
print("Batch size           :", BATCH_SIZE)
print("DataLoader workers   :", NUM_WORKERS)
print("Pin memory for CUDA  :", on_cuda)
print("Shuffle training     : True")
print("Shuffle evaluation   : False")

In [ ]:
# ============================================================
# Create DataLoaders
# ============================================================

train_loader = DataLoader(train_dataset, shuffle=True, generator=train_generator, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print(f"Training batches   : {len(train_loader):,}")
print(f"Validation batches : {len(val_loader):,}")
print(f"Test batches       : {len(test_loader):,}")

In [ ]:
# ============================================================
# Verify One Complete Training Batch
# ============================================================

# Restore the sampler state and CPU RNG after reading the demonstration batch.
sampler_state = train_generator.get_state()
try:
    with torch.random.fork_rng(devices=[]):
        batch_images, batch_labels, batch_image_ids = next(iter(train_loader))
finally:
    train_generator.set_state(sampler_state)

print("=" * 60)
print("TRAINING BATCH VERIFICATION")
print("=" * 60)
print("Batch image shape :", tuple(batch_images.shape))
print("Batch label shape :", tuple(batch_labels.shape))
print("Image dtype       :", batch_images.dtype)
print("Label dtype       :", batch_labels.dtype)
print("First eight labels:", batch_labels[:8].tolist())

assert batch_images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert batch_images.dtype == torch.float32 and batch_labels.dtype == torch.int64
assert len(batch_images) == len(batch_labels) == len(batch_image_ids)
assert len(batch_images) == min(BATCH_SIZE, len(train_dataset))

In [ ]:
# ============================================================
# Visualize Original and Augmented Training Images
# ============================================================

# Reverse normalization to show RGB intensities in the interval [0, 1].
mean_tensor = torch.tensor(image_mean).view(3, 1, 1)
std_tensor = torch.tensor(image_std).view(3, 1, 1)
augmentation_figure, axes = plt.subplots(4, 3, figsize=(9, 11))

with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(RANDOM_SEED)
    for row, class_index in enumerate((0, 1, 2, 7)):
        example = train_df.loc[train_df["class_index"] == class_index].iloc[0]
        with Image.open(example["cached_image_path"]) as cached_image:
            image = cached_image.convert("RGB")

        views = [evaluation_transform(image), train_transform(image), train_transform(image)]
        for col, (view, title) in enumerate(zip(views, ("Original", "Augmented A", "Augmented B"))):
            rgb = (view * std_tensor + mean_tensor).clamp(0, 1).permute(1, 2, 0)
            axes[row, col].imshow(rgb)
            axes[row, col].set_title(f"{CLASS_CODES[class_index]} · {title}")
            axes[row, col].axis("off")

augmentation_figure.tight_layout()
plt.show()
plt.close(augmentation_figure)

## C. Pretrained Model Preparation

In [ ]:
# ============================================================
# Load ImageNet Pretrained ResNet50
# ============================================================

initial_weights = ResNet50_Weights.IMAGENET1K_V2
if RESUME_TRAINING and artifact("stage1_latest.pth").is_file():
    # The latest checkpoint restores all model weights before training.
    initial_weights = None

model = resnet50(weights=initial_weights)
print("Architecture  :", "ResNet50")
print("Initial weights:", initial_weights if initial_weights is not None else "Resume this experiment")

In [ ]:
# ============================================================
# Inspect Original Classifier
# ============================================================

original_classifier = model.fc
in_features = original_classifier.in_features
print("Input features :", in_features)
print("Output features:", original_classifier.out_features)
print(original_classifier)

In [ ]:
# ============================================================
# Replace Final Classification Layer
# ============================================================

model.fc = nn.Linear(in_features, NUM_CLASSES)
print("New eight-class classifier:", model.fc)

In [ ]:
# ============================================================
# Freeze Pretrained Backbone
# ============================================================

model.requires_grad_(False)
model.fc.requires_grad_(True)
print("Trainable layer:", "fc")

In [ ]:
# ============================================================
# Move Model to Selected Device
# ============================================================

model = model.to(device)
print("Model device:", next(model.parameters()).device)

In [ ]:
# ============================================================
# Count Model Parameters
# ============================================================

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
frozen_parameters = total_parameters - trainable_parameters

print("=" * 60)
print("MODEL PARAMETER INFORMATION")
print("=" * 60)
print(f"Total parameters     : {total_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,}")
print(f"Frozen parameters    : {frozen_parameters:,}")
print(f"Trainable percentage : {100 * trainable_parameters / total_parameters:.2f}%")

In [ ]:
# ============================================================
# Inspect Trainable and Frozen Parameters
# ============================================================

trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
frozen_names = [name for name, parameter in model.named_parameters() if not parameter.requires_grad]
print("Trainable parameter tensors:", trainable_names)
print("Frozen parameter tensors   :", len(frozen_names))
print("Examples from the backbone :", frozen_names[:5])
assert set(trainable_names) == {"fc.weight", "fc.bias"}

In [ ]:
# ============================================================
# Inspect Model Architecture
# ============================================================

print("Top-level ResNet50 modules:")
for module_name, module in model.named_children():
    print(f"  {module_name:15s} {type(module).__name__}")
print("Classification layer:", model.fc)

In [ ]:
# ============================================================
# Test Forward Pass
# ============================================================

# Evaluation mode protects BatchNorm buffers during the dry run.
previous_mode = model.training
model.eval()
with torch.no_grad():
    sample_inputs = batch_images[:2].to(device)
    sample_outputs = model(sample_inputs)
model.train(previous_mode)

print("Input shape :", tuple(sample_inputs.shape))
print("Output shape:", tuple(sample_outputs.shape))
assert sample_outputs.shape == (len(sample_inputs), NUM_CLASSES)
assert torch.isfinite(sample_outputs).all()

In [ ]:
# ============================================================
# Calculate Class Weights
# ============================================================

class_counts = train_df["class_index"].value_counts().reindex(CLASS_IDS)
weights = len(train_df) / (NUM_CLASSES * class_counts.to_numpy())
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

weight_table = class_info[["class_code", "class_name"]].copy()
weight_table["training_count"] = class_counts.to_numpy()
weight_table["loss_weight"] = weights
display(weight_table)

In [ ]:
# ============================================================
# Initialize Weighted Cross-Entropy Loss
# ============================================================

criterion = nn.CrossEntropyLoss(weight=class_weights)
print(criterion)
print("Reported loss aggregation:", "Sample-count average of weighted batch means")

## D. Stage 1 — Transfer Learning / Feature Extraction

Train the classifier while keeping the pretrained backbone and its BatchNorm statistics fixed.
Select the best checkpoint by validation macro F1. The scheduler receives validation loss.
The training and validation functions below are also used during fine-tuning.

In [ ]:
# ============================================================
# Stage 1 Optimizer Configuration
# ============================================================

stage1_optimizer = torch.optim.AdamW(model.fc.parameters(), lr=STAGE1_LR, weight_decay=WEIGHT_DECAY)
print("Optimizer:", type(stage1_optimizer).__name__)
print("Classifier learning rate:", STAGE1_LR)
print("Weight decay:", WEIGHT_DECAY)

In [ ]:
# ============================================================
# Stage 1 Learning Rate Scheduler
# ============================================================

stage1_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    stage1_optimizer, mode="min", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE
)
print("Monitored value: validation loss")
print("Scheduler factor and patience:", SCHEDULER_FACTOR, SCHEDULER_PATIENCE)

In [ ]:
# ============================================================
# Stage 1 Training Configuration
# ============================================================

stage1_best_path = artifact("stage1_best.pth")
stage1_latest_path = artifact("stage1_latest.pth")
stage1_history_path = artifact("stage1_history.csv")
stage1_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Maximum epochs       :", STAGE1_EPOCHS)
print("Early-stop patience  :", STAGE1_PATIENCE)
print("Best checkpoint      :", stage1_best_path)
print("Latest checkpoint    :", stage1_latest_path)
print("Training history     :", stage1_history_path)

In [ ]:
# ============================================================
# Train One Epoch
# ============================================================

def train_one_epoch(model, train_loader, optimizer, scaler, freeze_backbone):
    model.train()
    if freeze_backbone:
        for layer in model.modules():
            if isinstance(layer, nn.BatchNorm2d):
                layer.eval()

    totals = torch.zeros(2, device=device)
    running_samples = 0

    for images, labels, _ in train_loader:
        images = images.to(device, non_blocking=on_cuda)
        labels = labels.to(device, non_blocking=on_cuda)
        optimizer.zero_grad(set_to_none=True)

        # ------------------------------------------------------------
        # Forward Pass and Optimization
        # ------------------------------------------------------------
        with torch.autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # ------------------------------------------------------------
        # Update Running Loss and Accuracy
        # ------------------------------------------------------------
        predictions = outputs.detach().argmax(dim=1)
        totals[0] += loss.detach() * len(labels)
        totals[1] += (predictions == labels).sum()
        running_samples += len(labels)

    running_loss, running_correct = totals.tolist()
    train_loss = running_loss / running_samples
    train_accuracy = running_correct / running_samples
    return train_loss, train_accuracy

In [ ]:
# ============================================================
# Validate One Epoch
# ============================================================

def validate_one_epoch(model, val_loader):
    model.eval()
    totals = torch.zeros(2, device=device)
    running_samples = 0
    true_labels = []
    predicted_labels = []

    with torch.no_grad():
        for images, batch_labels, _ in val_loader:
            images = images.to(device, non_blocking=on_cuda)
            labels = batch_labels.to(device, non_blocking=on_cuda)
            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            totals[0] += loss * len(labels)
            totals[1] += (predictions == labels).sum()
            running_samples += len(labels)
            true_labels.extend(batch_labels.tolist())
            predicted_labels.extend(predictions.cpu().tolist())

    running_loss, running_correct = totals.tolist()
    val_loss = running_loss / running_samples
    val_accuracy = running_correct / running_samples
    val_macro_f1 = f1_score(true_labels, predicted_labels, labels=CLASS_IDS, average="macro", zero_division=0)
    return val_loss, val_accuracy, val_macro_f1

In [ ]:
# ============================================================
# Checkpoint Save and Recovery
# ============================================================

def save_checkpoint(state, path):
    # Replace the previous checkpoint only after the new file is complete.
    temporary_path = path.with_suffix(".tmp")
    torch.save(state, temporary_path)
    temporary_path.replace(path)


def restore_training_checkpoint(path, model, optimizer, scheduler, scaler):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    torch.set_rng_state(checkpoint["torch_rng"])
    train_generator.set_state(checkpoint["loader_rng"])
    if on_cuda:
        torch.cuda.set_rng_state(checkpoint["device_rng"], device)
    elif device.type == "mps":
        torch.mps.set_rng_state(checkpoint["device_rng"])
    return checkpoint

In [ ]:
# ============================================================
# Save Stage 1 Experiment Configuration
# ============================================================

settings = {
    "architecture": "ResNet50",
    "imagenet_weights": "IMAGENET1K_V2",
    "dataset": "ISIC 2019",
    "split_sizes": expected_sizes,
    "class_to_index": class_to_index,
    "class_weights": weights.tolist(),
    "cache_size": CACHE_SIZE,
    "input_size": IMAGE_SIZE,
    "normalization_mean": list(image_mean),
    "normalization_std": list(image_std),
    "augmentation": {
        "horizontal_flip": HORIZONTAL_FLIP, "vertical_flip": VERTICAL_FLIP,
        "rotation_degrees": ROTATION_DEGREES, "rotation_interpolation": "bilinear",
        "reflection_padding": rotation_padding, "rotation_fill": list(rotation_fill),
        "brightness": BRIGHTNESS, "contrast": CONTRAST, "saturation": SATURATION,
    },
    "seed": RANDOM_SEED,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "optimizer": "AdamW",
    "weight_decay": WEIGHT_DECAY,
    "loss": "weighted cross-entropy, sample-count average of batch means",
    "scheduler": "ReduceLROnPlateau on validation loss",
    "scheduler_factor": SCHEDULER_FACTOR,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "selection": "higher validation macro F1 across both stages; Stage 1 on a tie",
    "stage1": {"epochs": STAGE1_EPOCHS, "patience": STAGE1_PATIENCE, "classifier_lr": STAGE1_LR},
    "stage2": {"epochs": STAGE2_EPOCHS, "patience": STAGE2_PATIENCE,
               "backbone_lr": BACKBONE_LR, "classifier_lr": CLASSIFIER_LR},
    "device_type": device.type,
    "cuda_amp": use_amp,
    "torch": str(torch.__version__),
    "torchvision": str(torchvision.__version__),
}

config_path = artifact("config.json")
config = {"settings": settings, "training_hardware": [hardware]}
if RESUME_TRAINING:
    config = json.loads(config_path.read_text())
    if config["settings"] != settings:
        raise ValueError("Resume with the same recipe, device type, and package versions.")
    if hardware not in config["training_hardware"]:
        config["training_hardware"].append(hardware)
else:
    if any(path.exists() and any(path.iterdir()) for path in (MODEL_DIR, RESULTS_DIR)):
        raise FileExistsError("Choose an empty output folder for a new experiment.")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
temporary_config = config_path.with_suffix(".tmp")
temporary_config.write_text(json.dumps(config, indent=2))
temporary_config.replace(config_path)
training_hardware = config["training_hardware"]

augmentation_figure.savefig(artifact("augmentation_preview.png"), dpi=150)

print("=" * 60)
print("STAGE 1 AND STAGE 2 EXPERIMENT CONFIGURATION")
print("=" * 60)
print("Saved configuration:", config_path)
print("Stage 1 settings   :", settings["stage1"])
print("Stage 2 settings   :", settings["stage2"])
print("Training hardware  :", training_hardware)

In [ ]:
# ============================================================
# Stage 1 Training History and Recovery
# ============================================================

stage1_history = []
stage1_best_val_f1 = -1.0
stage1_best_epoch = 0
stage1_epochs_without_improvement = 0
stage1_best_weights = None
stage1_start_epoch = 1

if RESUME_TRAINING and stage1_latest_path.is_file():
    checkpoint = restore_training_checkpoint(
        stage1_latest_path, model, stage1_optimizer, stage1_scheduler, stage1_scaler
    )
    stage1_history = checkpoint["history"]
    stage1_best_val_f1 = checkpoint["best_f1"]
    stage1_best_epoch = checkpoint["best_epoch"]
    stage1_epochs_without_improvement = checkpoint["stale_epochs"]
    stage1_best_weights = checkpoint["best_weights"]
    stage1_start_epoch = checkpoint["epoch"] + 1
    del checkpoint

    # Recover derived files if the interruption occurred after saving the latest checkpoint.
    save_checkpoint(stage1_best_weights, stage1_best_path)
    pd.DataFrame(stage1_history).to_csv(stage1_history_path, index=False)

print("First epoch to run :", stage1_start_epoch)
print("Previously recorded epochs:", len(stage1_history))

In [ ]:
# ============================================================
# Stage 1 Sanity Check
# ============================================================

assert frozen_parameters > 0
previous_mode = model.training
model.eval()
with torch.no_grad():
    check_images = batch_images[:2].to(device)
    check_labels = batch_labels[:2].to(device)
    check_outputs = model(check_images)
    check_loss = criterion(check_outputs, check_labels)
model.train(previous_mode)

assert check_outputs.shape == (len(check_images), NUM_CLASSES)
assert torch.isfinite(check_loss)
print("Output shape        :", tuple(check_outputs.shape))
print("Loss before backward:", check_loss.item())
print("This check used no backward pass or optimizer step.")

In [ ]:
# ============================================================
# Stage 1 Training Loop
# ============================================================

for epoch in range(stage1_start_epoch, STAGE1_EPOCHS + 1):
    if stage1_epochs_without_improvement >= STAGE1_PATIENCE:
        break

    # ------------------------------------------------------------
    # Start Epoch Timer
    # ------------------------------------------------------------
    epoch_start_time = time.perf_counter()

    # ------------------------------------------------------------
    # Training Phase
    # ------------------------------------------------------------
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, stage1_optimizer, stage1_scaler, freeze_backbone=True
    )

    # ------------------------------------------------------------
    # Validation Phase
    # ------------------------------------------------------------
    val_loss, val_accuracy, val_macro_f1 = validate_one_epoch(model, val_loader)
    if not np.isfinite([train_loss, val_loss, val_macro_f1]).all():
        raise FloatingPointError(f"Stage 1 epoch {epoch}: non-finite metrics.")

    # ------------------------------------------------------------
    # Current Learning Rates, Epoch Time, and Learning Rate Scheduler
    # ------------------------------------------------------------
    classifier_lr = stage1_optimizer.param_groups[-1]["lr"]
    backbone_lr = 0.0
    epoch_time = time.perf_counter() - epoch_start_time
    stage1_scheduler.step(val_loss)

    # ------------------------------------------------------------
    # Store Training History
    # ------------------------------------------------------------
    stage1_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1,
        "classifier_lr": classifier_lr,
        "backbone_lr": backbone_lr,
        "epoch_time_seconds": epoch_time,
    })

    # ------------------------------------------------------------
    # Track the Best Checkpoint
    # ------------------------------------------------------------
    improved = val_macro_f1 > stage1_best_val_f1
    if improved:
        stage1_best_val_f1 = val_macro_f1
        stage1_best_epoch = epoch
        stage1_epochs_without_improvement = 0
        stage1_best_weights = {
            name: value.detach().to("cpu", copy=True) for name, value in model.state_dict().items()
        }
    else:
        stage1_epochs_without_improvement += 1

    # ------------------------------------------------------------
    # Save Latest Recovery Checkpoint
    # ------------------------------------------------------------
    device_rng = None
    if on_cuda:
        device_rng = torch.cuda.get_rng_state(device)
    elif device.type == "mps":
        device_rng = torch.mps.get_rng_state()
    save_checkpoint({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": stage1_optimizer.state_dict(),
        "scheduler": stage1_scheduler.state_dict(),
        "scaler": stage1_scaler.state_dict(),
        "history": stage1_history,
        "best_f1": stage1_best_val_f1,
        "best_epoch": stage1_best_epoch,
        "best_weights": stage1_best_weights,
        "stale_epochs": stage1_epochs_without_improvement,
        "torch_rng": torch.get_rng_state(),
        "loader_rng": train_generator.get_state(),
        "device_rng": device_rng,
    }, stage1_latest_path)

    # ------------------------------------------------------------
    # Save Best Checkpoint and Training History
    # ------------------------------------------------------------
    if improved:
        save_checkpoint(stage1_best_weights, stage1_best_path)
    pd.DataFrame(stage1_history).to_csv(stage1_history_path, index=False)

    # ------------------------------------------------------------
    # Display Epoch Results
    # ------------------------------------------------------------
    print("=" * 60)
    print(f"STAGE 1 — EPOCH {epoch} / {STAGE1_EPOCHS}")
    print("=" * 60)
    print(f"Training loss      : {train_loss:.4f}")
    print(f"Training accuracy  : {train_accuracy:.2%}")
    print(f"Validation loss    : {val_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.2%}")
    print(f"Validation macro F1: {val_macro_f1:.4f}")
    print(f"Classifier LR      : {classifier_lr:.2e}")
    print(f"Epoch time         : {epoch_time:.1f} seconds")
    print("Best model updated :", "Yes" if improved else "No")
    print(f"Early-stop progress: {stage1_epochs_without_improvement} / {STAGE1_PATIENCE}")

    # ------------------------------------------------------------
    # Early Stopping
    # ------------------------------------------------------------
    if stage1_epochs_without_improvement >= STAGE1_PATIENCE:
        print("Early stopping: validation macro F1 did not improve.")
        break

# ------------------------------------------------------------
# Final Training Summary
# ------------------------------------------------------------
print(f"Stage 1 recorded epochs: {len(stage1_history)}")
print(f"Best epoch: {stage1_best_epoch} | Best validation macro F1: {stage1_best_val_f1:.4f}")

In [ ]:
# ============================================================
# Load Stage 1 Training History
# ============================================================

stage1_history_df = pd.read_csv(stage1_history_path, float_precision="round_trip")
display(stage1_history_df)

In [ ]:
# ============================================================
# Inspect Best Stage 1 Epoch
# ============================================================

stage1_best_row = stage1_history_df.loc[stage1_history_df["val_macro_f1"].idxmax()]
stage1_best_epoch = int(stage1_best_row["epoch"])
stage1_best_f1 = float(stage1_best_row["val_macro_f1"])
display(stage1_best_row.to_frame(name="Stage 1 best epoch"))

## E. Stage 2 — Fine-Tuning

Initialize from this run's best Stage 1 checkpoint. Unfreeze the backbone, allow BatchNorm
statistics to update, and use separate learning rates for the backbone and classification layer.

In [ ]:
# ============================================================
# Load Best Stage 1 Checkpoint
# ============================================================

model.load_state_dict(torch.load(stage1_best_path, map_location="cpu", weights_only=True))
print("Loaded Stage 1 checkpoint:", stage1_best_path)
print("Best validation macro F1 :", stage1_best_f1)

In [ ]:
# ============================================================
# Unfreeze Backbone
# ============================================================

model.requires_grad_(True)
assert all(parameter.requires_grad for parameter in model.parameters())

In [ ]:
# ============================================================
# Inspect Stage 2 Trainable Parameters
# ============================================================

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print("=" * 60)
print("STAGE 2 MODEL PARAMETER INFORMATION")
print("=" * 60)
print(f"Total parameters     : {total_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,}")
print(f"Frozen parameters    : {total_parameters - trainable_parameters:,}")

In [ ]:
# ============================================================
# Stage 2 Learning Rates
# ============================================================

print("Backbone learning rate  :", BACKBONE_LR)
print("Classifier learning rate:", CLASSIFIER_LR)
print("Weight decay            :", WEIGHT_DECAY)

In [ ]:
# ============================================================
# Stage 2 Optimizer
# ============================================================

backbone_parameters = [parameter for name, parameter in model.named_parameters() if not name.startswith("fc.")]
stage2_optimizer = torch.optim.AdamW([
    {"params": backbone_parameters, "lr": BACKBONE_LR},
    {"params": model.fc.parameters(), "lr": CLASSIFIER_LR},
], weight_decay=WEIGHT_DECAY)
print("Optimizer         :", type(stage2_optimizer).__name__)
print("Parameter group LR:", [group["lr"] for group in stage2_optimizer.param_groups])

In [ ]:
# ============================================================
# Stage 2 Learning Rate Scheduler
# ============================================================

stage2_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    stage2_optimizer, mode="min", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE
)
print("Monitored value: validation loss")
print("Scheduler factor and patience:", SCHEDULER_FACTOR, SCHEDULER_PATIENCE)

In [ ]:
# ============================================================
# Stage 2 Training Configuration
# ============================================================

stage2_best_path = artifact("stage2_best.pth")
stage2_latest_path = artifact("stage2_latest.pth")
stage2_history_path = artifact("stage2_history.csv")
stage2_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Maximum epochs       :", STAGE2_EPOCHS)
print("Early-stop patience  :", STAGE2_PATIENCE)
print("Best checkpoint      :", stage2_best_path)
print("Latest checkpoint    :", stage2_latest_path)
print("Training history     :", stage2_history_path)

In [ ]:
# ============================================================
# Stage 2 Training History and Recovery
# ============================================================

stage2_history = []
stage2_best_val_f1 = -1.0
stage2_best_epoch = 0
stage2_epochs_without_improvement = 0
stage2_best_weights = None
stage2_start_epoch = 1

if RESUME_TRAINING and stage2_latest_path.is_file():
    checkpoint = restore_training_checkpoint(
        stage2_latest_path, model, stage2_optimizer, stage2_scheduler, stage2_scaler
    )
    stage2_history = checkpoint["history"]
    stage2_best_val_f1 = checkpoint["best_f1"]
    stage2_best_epoch = checkpoint["best_epoch"]
    stage2_epochs_without_improvement = checkpoint["stale_epochs"]
    stage2_best_weights = checkpoint["best_weights"]
    stage2_start_epoch = checkpoint["epoch"] + 1
    del checkpoint

    # Recover derived files if the interruption occurred after saving the latest checkpoint.
    save_checkpoint(stage2_best_weights, stage2_best_path)
    pd.DataFrame(stage2_history).to_csv(stage2_history_path, index=False)

print("First epoch to run :", stage2_start_epoch)
print("Previously recorded epochs:", len(stage2_history))

In [ ]:
# ============================================================
# Stage 2 Sanity Check
# ============================================================

assert all(parameter.requires_grad for parameter in model.parameters())
previous_mode = model.training
model.eval()
with torch.no_grad():
    check_images = batch_images[:2].to(device)
    check_labels = batch_labels[:2].to(device)
    check_outputs = model(check_images)
    check_loss = criterion(check_outputs, check_labels)
model.train(previous_mode)

assert check_outputs.shape == (len(check_images), NUM_CLASSES)
assert torch.isfinite(check_loss)
print("Output shape        :", tuple(check_outputs.shape))
print("Loss before backward:", check_loss.item())
print("This check used no backward pass or optimizer step.")

In [ ]:
# ============================================================
# Save Stage 2 Fine-Tuning Configuration
# ============================================================

stage2_configuration = {
    **settings["stage2"],
    "starting_checkpoint": stage1_best_path.name,
    "optimizer": "AdamW",
    "weight_decay": WEIGHT_DECAY,
    "scheduler": settings["scheduler"],
    "scheduler_factor": SCHEDULER_FACTOR,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "cuda_amp": use_amp,
}
stage2_config_path = artifact("stage2_configuration.json")
stage2_config_path.write_text(json.dumps(stage2_configuration, indent=2))
print("Saved fine-tuning configuration:", stage2_config_path)

In [ ]:
# ============================================================
# Stage 2 Training Loop
# ============================================================

for epoch in range(stage2_start_epoch, STAGE2_EPOCHS + 1):
    if stage2_epochs_without_improvement >= STAGE2_PATIENCE:
        break

    # ------------------------------------------------------------
    # Start Epoch Timer
    # ------------------------------------------------------------
    epoch_start_time = time.perf_counter()

    # ------------------------------------------------------------
    # Training Phase
    # ------------------------------------------------------------
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, stage2_optimizer, stage2_scaler, freeze_backbone=False
    )

    # ------------------------------------------------------------
    # Validation Phase
    # ------------------------------------------------------------
    val_loss, val_accuracy, val_macro_f1 = validate_one_epoch(model, val_loader)
    if not np.isfinite([train_loss, val_loss, val_macro_f1]).all():
        raise FloatingPointError(f"Stage 2 epoch {epoch}: non-finite metrics.")

    # ------------------------------------------------------------
    # Current Learning Rates, Epoch Time, and Learning Rate Scheduler
    # ------------------------------------------------------------
    classifier_lr = stage2_optimizer.param_groups[-1]["lr"]
    backbone_lr = stage2_optimizer.param_groups[0]["lr"]
    epoch_time = time.perf_counter() - epoch_start_time
    stage2_scheduler.step(val_loss)

    # ------------------------------------------------------------
    # Store Training History
    # ------------------------------------------------------------
    stage2_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_macro_f1,
        "classifier_lr": classifier_lr,
        "backbone_lr": backbone_lr,
        "epoch_time_seconds": epoch_time,
    })

    # ------------------------------------------------------------
    # Track the Best Checkpoint
    # ------------------------------------------------------------
    improved = val_macro_f1 > stage2_best_val_f1
    if improved:
        stage2_best_val_f1 = val_macro_f1
        stage2_best_epoch = epoch
        stage2_epochs_without_improvement = 0
        stage2_best_weights = {
            name: value.detach().to("cpu", copy=True) for name, value in model.state_dict().items()
        }
    else:
        stage2_epochs_without_improvement += 1

    # ------------------------------------------------------------
    # Save Latest Recovery Checkpoint
    # ------------------------------------------------------------
    device_rng = None
    if on_cuda:
        device_rng = torch.cuda.get_rng_state(device)
    elif device.type == "mps":
        device_rng = torch.mps.get_rng_state()
    save_checkpoint({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": stage2_optimizer.state_dict(),
        "scheduler": stage2_scheduler.state_dict(),
        "scaler": stage2_scaler.state_dict(),
        "history": stage2_history,
        "best_f1": stage2_best_val_f1,
        "best_epoch": stage2_best_epoch,
        "best_weights": stage2_best_weights,
        "stale_epochs": stage2_epochs_without_improvement,
        "torch_rng": torch.get_rng_state(),
        "loader_rng": train_generator.get_state(),
        "device_rng": device_rng,
    }, stage2_latest_path)

    # ------------------------------------------------------------
    # Save Best Checkpoint and Training History
    # ------------------------------------------------------------
    if improved:
        save_checkpoint(stage2_best_weights, stage2_best_path)
    pd.DataFrame(stage2_history).to_csv(stage2_history_path, index=False)

    # ------------------------------------------------------------
    # Display Epoch Results
    # ------------------------------------------------------------
    print("=" * 60)
    print(f"STAGE 2 — EPOCH {epoch} / {STAGE2_EPOCHS}")
    print("=" * 60)
    print(f"Training loss      : {train_loss:.4f}")
    print(f"Training accuracy  : {train_accuracy:.2%}")
    print(f"Validation loss    : {val_loss:.4f}")
    print(f"Validation accuracy: {val_accuracy:.2%}")
    print(f"Validation macro F1: {val_macro_f1:.4f}")
    print(f"Classifier LR      : {classifier_lr:.2e}")
    print(f"Backbone LR        : {backbone_lr:.2e}")
    print(f"Epoch time         : {epoch_time:.1f} seconds")
    print("Best model updated :", "Yes" if improved else "No")
    print(f"Early-stop progress: {stage2_epochs_without_improvement} / {STAGE2_PATIENCE}")

    # ------------------------------------------------------------
    # Early Stopping
    # ------------------------------------------------------------
    if stage2_epochs_without_improvement >= STAGE2_PATIENCE:
        print("Early stopping: validation macro F1 did not improve.")
        break

# ------------------------------------------------------------
# Final Training Summary
# ------------------------------------------------------------
print(f"Stage 2 recorded epochs: {len(stage2_history)}")
print(f"Best epoch: {stage2_best_epoch} | Best validation macro F1: {stage2_best_val_f1:.4f}")

In [ ]:
# ============================================================
# Load Stage 2 Training History
# ============================================================

stage2_history_df = pd.read_csv(stage2_history_path, float_precision="round_trip")
display(stage2_history_df)

In [ ]:
# ============================================================
# Inspect Best Stage 2 Epoch
# ============================================================

stage2_best_row = stage2_history_df.loc[stage2_history_df["val_macro_f1"].idxmax()]
stage2_best_epoch = int(stage2_best_row["epoch"])
stage2_best_f1 = float(stage2_best_row["val_macro_f1"])
display(stage2_best_row.to_frame(name="Stage 2 best epoch"))

## Final Model Selection and Test Evaluation

Compare the two best checkpoints using validation macro F1. Keep the Stage 1 checkpoint on an exact tie. Once selection is complete, evaluate the chosen model on the permanent test split.

In [ ]:
# ============================================================
# Compare Validation Results and Select Final Checkpoint
# ============================================================

stage1_seconds = float(stage1_history_df["epoch_time_seconds"].sum())
stage2_seconds = float(stage2_history_df["epoch_time_seconds"].sum())

stage_summary = pd.DataFrame([
    {"stage": "stage1", "epochs": len(stage1_history_df), "best_epoch": stage1_best_epoch,
     "best_val_macro_f1": stage1_best_f1, "minutes": stage1_seconds / 60},
    {"stage": "stage2", "epochs": len(stage2_history_df), "best_epoch": stage2_best_epoch,
     "best_val_macro_f1": stage2_best_f1, "minutes": stage2_seconds / 60},
])
display(stage_summary)
stage_summary.to_csv(artifact("stage_summary.csv"), index=False)

selected_stage = "stage1"
selected_path = stage1_best_path
selected_epoch = stage1_best_epoch
selected_validation_macro_f1 = stage1_best_f1

if stage2_best_f1 > stage1_best_f1:
    selected_stage = "stage2"
    selected_path = stage2_best_path
    selected_epoch = stage2_best_epoch
    selected_validation_macro_f1 = stage2_best_f1

fine_tuning_delta = stage2_best_f1 - stage1_best_f1
print(f"Fine-tuning change      : {100 * fine_tuning_delta:+.2f} percentage points")
print("Selected checkpoint     :", selected_path.name)
print("Selected epoch          :", selected_epoch)

In [ ]:
# ============================================================
# Load Selected Checkpoint
# ============================================================

best_model_state = torch.load(selected_path, map_location="cpu", weights_only=True)
model.load_state_dict(best_model_state)
del best_model_state
model.eval()

print("Model device                 :", next(model.parameters()).device)
print(f"Selected validation macro F1 : {selected_validation_macro_f1:.4f}")

In [ ]:
# ============================================================
# Evaluate the Held-Out Test Set
# ============================================================

test_running_loss = torch.zeros((), device=device)
test_running_correct = torch.zeros((), device=device)
test_running_samples = 0

all_test_labels = []
all_test_predictions = []
all_test_probabilities = []
all_test_image_ids = []

model.eval()

# ------------------------------------------------------------
# Forward Pass on the Test Split
# ------------------------------------------------------------
with torch.no_grad():
    for images, labels, image_ids in test_loader:
        images = images.to(device, non_blocking=on_cuda)
        labels = labels.to(device, non_blocking=on_cuda)

        outputs = model(images)
        loss = criterion(outputs, labels)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)

        batch_size = labels.size(0)
        test_running_loss += loss * batch_size
        test_running_correct += (predictions == labels).sum()
        test_running_samples += batch_size

        all_test_labels.extend(labels.cpu().tolist())
        all_test_predictions.extend(predictions.cpu().tolist())
        all_test_probabilities.extend(probabilities.cpu().tolist())
        all_test_image_ids.extend(image_ids)

# ------------------------------------------------------------
# Check Results and Compute Aggregate Loss and Accuracy
# ------------------------------------------------------------
all_test_probabilities = np.asarray(all_test_probabilities)
test_loss = test_running_loss.item() / test_running_samples
test_accuracy = test_running_correct.item() / test_running_samples

assert test_running_samples == len(test_df)
assert all_test_probabilities.shape == (len(test_df), len(CLASS_IDS))
if not np.isfinite(test_loss) or not np.isfinite(all_test_probabilities).all():
    raise FloatingPointError("Test loss or class scores are non-finite.")

print("Selected checkpoint :", selected_path.name)
print("Test images scored  :", test_running_samples)
print(f"Test loss           : {test_loss:.4f}")
print(f"Test accuracy       : {test_accuracy:.2%}")

## Training History

Read the recorded curves for each stage on its own epoch axis. A lower validation loss and a higher validation macro F1 can occur at different epochs. The checkpoint rule uses macro F1.

In [ ]:
# ============================================================
# Plot Training and Validation Loss
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for stage, history, ax in zip(("Stage 1", "Stage 2"), (stage1_history_df, stage2_history_df), axes):
    ax.plot(history["epoch"], history["train_loss"], marker="o", label="Training")
    ax.plot(history["epoch"], history["val_loss"], marker="o", label="Validation")
    ax.set(title=f"{stage} — Loss", xlabel="Epoch", ylabel="Weighted cross-entropy")
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
fig.savefig(artifact("loss_curves.png"), dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Plot Training and Validation Accuracy
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for stage, history, ax in zip(("Stage 1", "Stage 2"), (stage1_history_df, stage2_history_df), axes):
    ax.plot(history["epoch"], 100 * history["train_accuracy"], marker="o", label="Training")
    ax.plot(history["epoch"], 100 * history["val_accuracy"], marker="o", label="Validation")
    ax.set(title=f"{stage} — Accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
fig.savefig(artifact("accuracy_curves.png"), dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Plot Validation Macro F1
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for stage, history, ax in zip(("Stage 1", "Stage 2"), (stage1_history_df, stage2_history_df), axes):
    ax.plot(history["epoch"], history["val_macro_f1"], marker="o", label="Validation")
    ax.set(title=f"{stage} — Macro F1", xlabel="Epoch", ylabel="Macro F1")
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
fig.savefig(artifact("validation_macro_f1.png"), dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Save Combined Training Curves
# ============================================================

# Assemble the three existing figures into one image for the experiment record.
curve_panels = []

for filename in ("loss_curves.png", "accuracy_curves.png", "validation_macro_f1.png"):
    with Image.open(artifact(filename)) as image:
        curve_panels.append(image.convert("RGB"))

panel_width = max(panel.width for panel in curve_panels)
total_height = sum(panel.height for panel in curve_panels)
combined_curves = Image.new("RGB", (panel_width, total_height), "white")

vertical_position = 0
for panel in curve_panels:
    combined_curves.paste(panel, (0, vertical_position))
    vertical_position += panel.height

combined_curves.save(artifact("training_curves.png"))
print("Combined curves saved to:", artifact("training_curves.png"))

## Classification Results and Error Analysis

Report class-level performance in the permanent label order. Use counts to see how many predictions fall in each pair of classes, and normalize each row to see the fraction of a true class assigned to each prediction.

In [ ]:
# ============================================================
# Detailed Classification Report
# ============================================================

report = classification_report(
    all_test_labels, all_test_predictions, labels=CLASS_IDS,
    target_names=class_names, zero_division=0, output_dict=True,
)

test_macro_precision = report["macro avg"]["precision"]
test_macro_recall = report["macro avg"]["recall"]
test_macro_f1 = report["macro avg"]["f1-score"]
test_weighted_f1 = report["weighted avg"]["f1-score"]

print(f"{'Class':<27} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
for name in class_names + ["macro avg", "weighted avg"]:
    row = report[name]
    print(f"{name:<27} {row['precision']:>10.4f} {row['recall']:>10.4f} "
          f"{row['f1-score']:>10.4f} {int(row['support']):>10}")

In [ ]:
# ============================================================
# Convert Classification Report to DataFrame
# ============================================================

report_table = pd.DataFrame(report).transpose().drop(index="accuracy")
display(report_table)

In [ ]:
# ============================================================
# Raw Confusion Matrix
# ============================================================

matrix = confusion_matrix(all_test_labels, all_test_predictions, labels=CLASS_IDS)
raw_matrix_table = pd.DataFrame(matrix, index=CLASS_CODES, columns=CLASS_CODES)

display(raw_matrix_table)

In [ ]:
# ============================================================
# Plot Raw Confusion Matrix
# ============================================================

fig, ax = plt.subplots(figsize=(8, 7))
heatmap = ax.imshow(matrix, cmap="Blues")
fig.colorbar(heatmap, ax=ax, label="Number of images")
ax.set_xticks(CLASS_IDS, CLASS_CODES)
ax.set_yticks(CLASS_IDS, CLASS_CODES)
ax.set(title="Held-out confusion matrix — counts", xlabel="Predicted class", ylabel="True class")

text_threshold = matrix.max() / 2
for row in CLASS_IDS:
    for col in CLASS_IDS:
        count = matrix[row, col]
        text_color = "white" if count > text_threshold else "black"
        ax.text(col, row, str(count), ha="center", va="center", color=text_color)

fig.tight_layout()
fig.savefig(artifact("confusion_matrix_raw.png"), dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Normalized Confusion Matrix
# ============================================================

row_totals = matrix.sum(axis=1, keepdims=True)
normalized = np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)
normalized_matrix_table = pd.DataFrame(normalized, index=CLASS_CODES, columns=CLASS_CODES)

print("Rows show the fraction of each true class assigned to each predicted class.")
display(normalized_matrix_table.round(3))

In [ ]:
# ============================================================
# Plot Normalized Confusion Matrix
# ============================================================

fig, ax = plt.subplots(figsize=(8, 7))
heatmap = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1)
fig.colorbar(heatmap, ax=ax, label="Fraction of true class")
ax.set_xticks(CLASS_IDS, CLASS_CODES)
ax.set_yticks(CLASS_IDS, CLASS_CODES)
ax.set(title="Held-out confusion matrix — normalized", xlabel="Predicted class", ylabel="True class")

for row in CLASS_IDS:
    for col in CLASS_IDS:
        value = normalized[row, col]
        text_color = "white" if value >= 0.5 else "black"
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=text_color)

fig.tight_layout()
fig.savefig(artifact("confusion_matrix.png"), dpi=150)
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Per-Class Performance Table
# ============================================================

per_class = report_table.loc[class_names]
display(per_class)

### Interpreting Prediction Confidence

Predicted confidence is the raw softmax score assigned to the predicted class. Use it to rank model errors for inspection. Estimates of calibrated probability and clinical certainty require separate evidence.

In [ ]:
# ============================================================
# Predicted Confidence Scores
# ============================================================

test_confidences = all_test_probabilities.max(axis=1)
correct_predictions = np.asarray(all_test_labels) == np.asarray(all_test_predictions)

confidence_table = pd.DataFrame({"correct": correct_predictions, "confidence": test_confidences})
display(confidence_table.groupby("correct")["confidence"].agg(["count", "mean", "median", "max"]))

In [ ]:
# ============================================================
# Create Test Prediction Table
# ============================================================

test_predictions = pd.DataFrame({
    "image": all_test_image_ids,
    "true_class": [CLASS_CODES[index] for index in all_test_labels],
    "predicted_class": [CLASS_CODES[index] for index in all_test_predictions],
})

class_scores = pd.DataFrame(all_test_probabilities, columns=[f"probability_{code}" for code in CLASS_CODES])
test_predictions = pd.concat([test_predictions, class_scores], axis=1)
test_predictions["confidence"] = test_confidences
test_predictions["correct"] = correct_predictions

display(test_predictions.head())

In [ ]:
# ============================================================
# Attach Image Information to Test Predictions
# ============================================================

test_image_info = test_df[["image", "cached_image_path"]]
test_predictions_with_images = test_predictions.merge(test_image_info, on="image", validate="one_to_one")
assert len(test_predictions_with_images) == len(test_predictions)
assert test_predictions_with_images["cached_image_path"].notna().all()

misclassified_df = test_predictions_with_images.loc[~test_predictions_with_images["correct"]].copy()

print("Predictions matched to cached images :", len(test_predictions_with_images))
print("Misclassified test images            :", len(misclassified_df))
display(misclassified_df[["image", "true_class", "predicted_class", "confidence"]].head())

In [ ]:
# ============================================================
# Visualize Random Misclassified Test Images
# ============================================================

if misclassified_df.empty:
    print("The selected checkpoint made no errors on this test split.")
else:
    examples = misclassified_df.sample(n=min(8, len(misclassified_df)), random_state=RANDOM_SEED)

    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax in axes.flat:
        ax.axis("off")

    for ax, example in zip(axes.flat, examples.itertuples(index=False)):
        with Image.open(example.cached_image_path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(
            f"True: {example.true_class} | Predicted: {example.predicted_class}" + "\n"
            + f"Softmax score: {example.confidence:.1%}",
            fontsize=10,
        )

    fig.suptitle("Random Misclassified Test Images")
    fig.tight_layout()
    fig.savefig(artifact("misclassified_random.png"), dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# Visualize Highest-Confidence Misclassified Test Images
# ============================================================

if misclassified_df.empty:
    print("The selected checkpoint made no errors on this test split.")
else:
    examples = misclassified_df.nlargest(8, "confidence")

    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax in axes.flat:
        ax.axis("off")

    for ax, example in zip(axes.flat, examples.itertuples(index=False)):
        with Image.open(example.cached_image_path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(
            f"True: {example.true_class} | Predicted: {example.predicted_class}" + "\n"
            + f"Softmax score: {example.confidence:.1%}",
            fontsize=10,
        )

    fig.suptitle("Highest-Confidence Misclassified Test Images")
    fig.tight_layout()
    fig.savefig(artifact("misclassified_high_confidence.png"), dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# Save Final Test Results
# ============================================================

summary = {
    "selected_stage": selected_stage,
    "selected_checkpoint": selected_path.name,
    "selected_epoch": int(selected_epoch),
    "selected_validation_macro_f1": float(selected_validation_macro_f1),
    "best_validation_macro_f1": float(selected_validation_macro_f1),
    "stage1_best_validation_macro_f1": float(stage1_best_f1),
    "stage2_best_validation_macro_f1": float(stage2_best_f1),
    "best_stage2_validation_macro_f1": float(stage2_best_f1),
    "fine_tuning_validation_f1_change_pp": 100 * float(fine_tuning_delta),
    "epochs_completed": {"stage1": len(stage1_history_df), "stage2": len(stage2_history_df)},
    "training_and_validation_seconds": stage1_seconds + stage2_seconds,
    "training_minutes": (stage1_seconds + stage2_seconds) / 60,
    "total_parameters": int(total_parameters),
    "training_hardware": training_hardware,
    "training_precision": "CUDA AMP" if use_amp else "float32",
    "evaluation_hardware": hardware,
    "test_samples": test_running_samples,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "test_macro_precision": test_macro_precision,
    "test_macro_recall": test_macro_recall,
    "test_macro_f1": test_macro_f1,
    "test_weighted_f1": test_weighted_f1,
}

artifact("test_results.json").write_text(json.dumps(summary, indent=2))
report_table.to_csv(artifact("classification_report.csv"), index_label="label")
per_class.to_csv(artifact("per_class_results.csv"), index_label="class")
raw_matrix_table.to_csv(artifact("confusion_matrix.csv"))
normalized_matrix_table.to_csv(artifact("confusion_matrix_normalized.csv"))
test_predictions_with_images.to_csv(artifact("test_predictions.csv"), index=False)
misclassified_df.to_csv(artifact("misclassified_predictions.csv"), index=False)

print("Test metrics saved to :", artifact("test_results.json"))

In [ ]:
# ============================================================
# Final Experiment Summary
# ============================================================

print("=" * 60)
print("EXPERIMENT 05 — FINAL SUMMARY")
print("=" * 60)
print("Selected checkpoint      :", selected_path.name)
print(f"Validation macro F1      : {selected_validation_macro_f1:.4f}")
print(f"Test loss                : {test_loss:.4f}")
print(f"Test accuracy            : {test_accuracy:.2%}")
print(f"Test macro precision     : {test_macro_precision:.4f}")
print(f"Test macro recall        : {test_macro_recall:.4f}")
print(f"Test macro F1             : {test_macro_f1:.4f}")
print(f"Test weighted F1          : {test_weighted_f1:.4f}")
print("Test images               :", test_running_samples)
print("Misclassified test images :", len(misclassified_df))
print("Model output directory    :", MODEL_DIR)
print("Results output directory  :", RESULTS_DIR)

### Compare with Experiment 3

Read historical baseline files only after training and final evaluation. The score deltas are Experiment 5 minus Experiment 3 in percentage points, and the time delta is in minutes. Training minutes sum the recorded training and validation epochs. Leave missing baseline values blank. Run time is hardware-dependent. If Stage 1 wins here, the held-out comparison also reflects a different stage-selection rule.

In [ ]:
# ============================================================
# Compare With Experiment 03
# ============================================================

baseline_summary_path = BASELINE_DIR / "resnet50_transfer_test_results.json"
baseline_report_path = BASELINE_DIR / "resnet50_transfer_per_class_results.csv"
baseline_config_path = BASELINE_DIR / "resnet50_transfer_stage2_config.json"
if not baseline_config_path.is_file():
    baseline_config_path = BASELINE_DIR / "public_configs" / baseline_config_path.name
baseline_history_paths = [
    BASELINE_DIR / f"resnet50_transfer_stage{stage}_history.csv" for stage in (1, 2)
]

baseline = json.loads(baseline_summary_path.read_text()) if baseline_summary_path.is_file() else {}
baseline_report = pd.read_csv(baseline_report_path, index_col=0) if baseline_report_path.is_file() else None
baseline_config = json.loads(baseline_config_path.read_text()) if baseline_config_path.is_file() else {}
baseline_histories = [pd.read_csv(path) if path.is_file() else None for path in baseline_history_paths]
baseline_minutes = None
if all(history is not None and "epoch_time_seconds" in history for history in baseline_histories):
    baseline_minutes = sum(history["epoch_time_seconds"].sum() for history in baseline_histories) / 60

stage_scores = []
for stage, history in enumerate(baseline_histories, start=1):
    score = baseline.get(f"best_stage{stage}_validation_macro_f1")
    if score is None and history is not None and "val_macro_f1" in history:
        score = history["val_macro_f1"].max()
    stage_scores.append(score)

baseline_best = max(stage_scores) if all(pd.notna(score) for score in stage_scores) else None
baseline_hardware = baseline_config.get("device")
if baseline_hardware == "mps":
    baseline_hardware = "Apple Silicon (MPS)"

comparison_rows = [
    ("Best validation macro F1 across stages", baseline_best, summary["best_validation_macro_f1"], "pp"),
    ("Best Stage 2 validation macro F1", stage_scores[1], stage2_best_f1, "pp"),
    ("Test accuracy", baseline.get("test_accuracy"), summary["test_accuracy"], "pp"),
    ("Test macro F1", baseline.get("test_macro_f1"), summary["test_macro_f1"], "pp"),
    ("Test weighted F1", baseline.get("test_weighted_f1"), summary["test_weighted_f1"], "pp"),
    ("Training minutes", baseline_minutes, summary["training_minutes"], "min"),
]
for name in class_names:
    baseline_f1 = None
    if baseline_report is not None and name in baseline_report.index and "f1-score" in baseline_report:
        baseline_f1 = baseline_report.loc[name, "f1-score"]
    comparison_rows.append((f"Test F1 · {name}", baseline_f1, report[name]["f1-score"], "pp"))

has_baseline = baseline_hardware is not None or any(pd.notna(row[1]) for row in comparison_rows)
if has_baseline:
    comparison = []
    for label, old, new, unit in comparison_rows:
        scale = 100 if unit == "pp" else 1
        old_value = old * scale if pd.notna(old) else None
        new_value = new * scale
        delta = new_value - old_value if old_value is not None else None
        comparison.append({
            "metric": label, "experiment_3": old_value, "experiment_5": new_value,
            "difference": delta, "value_unit": "%" if unit == "pp" else "min", "difference_unit": unit,
        })
    comparison.append({
        "metric": "Training hardware", "experiment_3": baseline_hardware,
        "experiment_5": ", ".join(config["training_hardware"]),
    })
    comparison = pd.DataFrame(comparison)
    comparison.to_csv(artifact("baseline_comparison.csv"), index=False)
    display(comparison)
    if baseline_hardware and set(config["training_hardware"]) != {baseline_hardware}:
        print("Training hardware differs. Treat the time comparison as descriptive.")
    if selected_stage == "stage1":
        print("Experiment 5 selected Stage 1. Experiment 3's test results used Stage 2.")
else:
    print("No baseline artifacts found. Experiment 5 results are saved independently.")

print(f"Models: {MODEL_DIR.resolve()} | Results: {RESULTS_DIR.resolve()}")

## Appendix — Running on an Online GPU Service

1. Upload the notebook and upload or mount the existing `dataset/splits/` and `dataset/ISIC_2019_256/` folders. Keep the directory structure and the permanent split CSVs.
2. Select an NVIDIA GPU runtime with compatible CUDA-enabled PyTorch (2.4 or later) and TorchVision builds. Check that the device cell prints `cuda` and the GPU name.
3. Set `PROJECT_DIR` to the mounted project folder. Set `OUTPUT_DIR` to writable persistent storage and leave `MODEL_DIR` and `RESULTS_DIR` pointing there by default.
4. For a fresh run, choose an empty output location, leave `RESUME_TRAINING = False`, and run the notebook from the top. TorchVision may fetch the ImageNet weights if they are absent from the runtime cache.
5. After an interruption, reconnect to the same storage with the same paths, training settings, split files, device type, and package versions. Set `RESUME_TRAINING = True` and run from the top. Training continues from the latest completed epoch.
6. Download the model checkpoints from `MODEL_DIR` and the histories, figures, configuration, and evaluation files from `RESULTS_DIR`.


For the optional Experiment 03 comparison, also mount its saved results at `BASELINE_DIR`.